# Проект: Детектирование медицинских масок на лицах

## 🎯 Цель
Разработать и сравнить два детектора объектов (YOLOv8 и Faster R-CNN) для выявления людей с масками и без. Оценить качество с помощью метрик mAP и IoU.

## 📊 Данные
Датасет из ~850 изображений. Разметка в формате Pascal VOC XML была конвертирована в формат YOLO (TXT). 
Классы: `with_mask`, `without_mask`, `mask_weared_incorrect`.

In [ ]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import glob

In [ ]:
from ultralytics import YOLO

# Загружаем предобученную модель (nano версия)
model = YOLO('yolov8n.pt')

# Запускаем обучение
results = model.train(
    data='D:/cv/mask_yolo/data.yaml',
    epochs=20,
    imgsz=640,
    device='cpu',
    batch=8,
    patience=5
)

In [ ]:
import glob
import matplotlib.pyplot as plt
import cv2

# 1. Загружаем лучшую обученную модель
best_model_path = 'D:/cv/mask_detection_project/runs/detect/train/weights/best.pt'
model = YOLO(best_model_path)

# 2. Берем несколько случайных картинок из валидационной выборки
val_images = glob.glob('D:/cv/mask_yolo/images/val/*.png')
sample_images = val_images[:4] # Берем 4 картинки для примера

# 3. Делаем предсказание и рисуем
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    # Предсказание
    results = model(img_path)
    
    # Рисуем рамки на картинке
    res_plotted = results[0].plot()
    
    # Конвертируем BGR в RGB для matplotlib
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(res_rgb)
    axes[i].set_title(f"Файл: {os.path.basename(img_path)}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
from PIL import Image
import os
import glob
from torch.utils.data import Dataset, DataLoader

# 1. Создаем кастомный Dataset для PyTorch
class MaskDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.imgs = list(sorted(os.listdir(img_dir)))

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, self.imgs[idx].replace('.png', '.txt'))
        
        img = Image.open(img_path).convert("RGB")
        
        # Читаем YOLO-разметку и конвертируем в формат PyTorch (xmin, ymin, xmax, ymax)
        boxes = []
        labels = []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id, xc, yc, w, h = map(float, parts)
                        # Переводим в абсолютные координаты
                        img_w, img_h = img.size
                        xmin = (xc - w/2) * img_w
                        ymin = (yc - h/2) * img_h
                        xmax = (xc + w/2) * img_w
                        ymax = (yc + h/2) * img_h
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(int(cls_id) + 1) # +1 так как 0 зарезервирован для фона
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        
        if self.transforms:
            img = self.transforms(img)
            
        return img, target

    def __len__(self):
        return len(self.imgs)

# 2. Трансформации (перевод в тензор)
transform = transforms.Compose([transforms.ToTensor()])

# 3. Загружаем данные
train_dataset = MaskDataset(
    img_dir='D:/cv/mask_yolo/images/train',
    label_dir='D:/cv/mask_yolo/labels/train',
    transforms=transform
)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

# 4. Создаем модель Faster R-CNN (MobileNet backbone для скорости)
model_frcnn = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights='DEFAULT')
num_classes = 4 # 1 фон + 3 класса
in_features = model_frcnn.roi_heads.box_predictor.cls_score.in_features
model_frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# 5. Обучение (упрощенный цикл на 2 эпохи для демонстрации)
device = torch.device('cpu')
model_frcnn.to(device)
params = [p for p in model_frcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

print("Начинаем обучение Faster R-CNN (2 эпохи)...")
model_frcnn.train()

for epoch in range(2):
    for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model_frcnn(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
    print(f"Эпоха {epoch+1} завершена, Loss: {losses.item():.4f}")

print("Обучение Faster R-CNN завершено!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image
import numpy as np

# 1. Переводим модель в режим инференса (оценки)
model_frcnn.eval()

# 2. Берем 4 картинки из валидации
val_images = glob.glob('D:/cv/mask_yolo/images/val/*.png')[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

classes = ['with_mask', 'without_mask', 'mask_weared_incorrect']

with torch.no_grad(): # Отключаем подсчет градиентов
    for i, img_path in enumerate(val_images):
        # Загружаем картинку
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0) # Добавляем размерность батча
        
        # Делаем предсказание
        prediction = model_frcnn(img_tensor)[0]
        
        # Рисуем картинку
        axes[i].imshow(img)
        
        # Рисуем рамки (берем только те, где уверенность > 0.5)
        for box, label, score in zip(prediction['boxes'], prediction['labels'], prediction['scores']):
            if score > 0.5:
                xmin, ymin, xmax, ymax = box.numpy()
                # Рисуем прямоугольник
                rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, 
                                         linewidth=2, edgecolor='r', facecolor='none')
                axes[i].add_patch(rect)
                
                # Подписываем класс
                cls_name = classes[label.item()-1] # -1, так как 0 это фон
                axes[i].text(xmin, ymin-5, f"{cls_name} {score:.2f}", 
                             color='white', fontsize=8, bbox=dict(facecolor='red', alpha=0.5))
        
        axes[i].axis('off')
        axes[i].set_title(os.path.basename(img_path))

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Пути к графикам (с приставкой Box)
base_path = 'D:/cv/mask_detection_project/runs/detect/train/'
graphs = {
    'Precision-Recall кривая (BoxPR)': 'BoxPR_curve.png',
    'F1-мера (BoxF1)': 'BoxF1_curve.png',
    'Матрица ошибок (Confusion Matrix)': 'confusion_matrix.png',
    'Общие результаты обучения': 'results.png'
}

for title, filename in graphs.items():
    img_path = os.path.join(base_path, filename)
    if os.path.exists(img_path):
        plt.figure(figsize=(10, 6))
        img = Image.open(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14)
        plt.show()
    else:
        print(f"Файл {filename} не найден.")

## 📊 Результаты и выводы

В ходе проекта были обучены и сравнены два детектора объектов для задачи распознавания медицинских масок:
1. **YOLOv8n** (одноуровневый, one-stage).
2. **Faster R-CNN** (двухуровневый, two-stage, с backbone MobileNetV3).

### Метрики качества (YOLOv8n):
*   **mAP50 (общий): 0.821**.
*   **with_mask: 0.943** (модель отлично находит людей в масках).
*   **without_mask: 0.821**.
*   **mask_weared_incorrect: 0.699** (низкий показатель из-за малого количества примеров в датасете).
*   Оптимальный порог уверенности (Confidence) для максимизации F1-меры: **0.361**.

### Сравнение архитектур:
| Характеристика | YOLOv8n | Faster R-CNN |
| :--- | :--- | :--- |
| **Тип** | Одноуровневый (One-stage) | Двухуровневый (Two-stage) |
| **Скорость** | **~30-60 мс** (реальное время) | ~200-500 мс (медленнее) |
| **Размер** | 6.5 МБ | 74 МБ |
| **Качество** | Отличное для реального времени | Высокое, но требует больше ресурсов |

### Вывод для бизнеса:
Для системы контроля ношения масок в реальном времени (например, на входе в офис или метро) **оптимальным выбором является YOLOv8** благодаря высокой скорости и малому размеру. Faster R-CNN показывает хорошее качество, но его скорость и вес делают его менее пригодным для работы на слабом оборудовании (например, на камерах видеонаблюдения).